# 04. Advanced Parsing with Knowledge

**Purpose:** Capstone notebook for *Regional Complexity* (Task 4). You will compare a **single-stage** address parser against a **two-stage + knowledge base (KB)** pipeline.

**Key idea:** Some problems require **system design** (retrieve the right rules/context), not just better prompts.

## Step 0: Set up environment & authenticate

In [ ]:
!git clone -b costs https://github.com/alxefremov/esmt-workshop.git

from pathlib import Path
import sys
PROJECT_ROOT = [it for it in [Path("src"), Path('esmt-workshop/src')] if (it / 'esmt_workshop').exists()][0].parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('PROJECT_ROOT =', PROJECT_ROOT)
%pip install -q -r {PROJECT_ROOT}/requirements.txt

import pandas as pd
import os
from esmt_workshop.evaluation import evaluate_predictions, save_evaluation_report
from esmt_workshop.experiment_logging import load_experiment_history, log_experiment_run
from esmt_workshop.student_api import process_batch_addresses, process_single_address
from esmt_workshop.authenticate import authenticate

STUDENT_EMAIL = authenticate()
from esmt_workshop.preset_params import get_preset_params
preset_params = get_preset_params()

## What you are learning

Global address parsing is hard because **formats vary by country** (postal code placement, state/province conventions, locality hierarchy, etc.).

You will compare two approaches:

1. **No country model (single-stage):**
   - `stage='advanced'`
   - No country detection and no KB rule retrieval.

2. **Country model enabled (two-stage + KB):**
   - `stage='two_stage'`
   - Pipeline:
     1) detect country using `COUNTRY_MODEL`
     2) retrieve country-specific formatting rules from a knowledge base
     3) parse using those rules ("RAG for rules")

**Key takeaway:** Better results sometimes come from **adding the right context** (retrieval + rules), not from making the prompt longer.

## Step 1: Limited Run (edit **one cell only**)

**How to use the next cell (do this in order):**

1. Paste your **best prompt** from Notebook 02 into `STUDENT_PROMPT_TEMPLATE`.
2. **Run A — No country model (single-stage):** set `COUNTRY_MODEL = ''` and run.
3. **Run B — Two-stage KB:** set `COUNTRY_MODEL = FLASH_COUNTRY_MODEL` and run again.

**Rule (limited run):** Only change:
- `COUNTRY_MODEL` (empty vs flash)
- your prompt text (`STUDENT_PROMPT_TEMPLATE`)

Do not change other parameters in this limited-run cell.

In [ ]:
import time

# -----------------------------
# PROMPT (paste your best prompt here)
# -----------------------------
STUDENT_PROMPT_TEMPLATE = '''You are an AML address parser.

Input address:
{address}

Return strict JSON only with schema:
{schema}

Rules:
- Extract Town Name, Postal Code, Remaining Address, Country Code (2 characters).
- Town Name must be city/locality only.
- Postal Code must contain only postal tokens.
- Country Code must be ISO alpha-2 uppercase.
- Do not invent values.
- Use empty strings if unknown.
'''

PROMPT_TO_RUN = STUDENT_PROMPT_TEMPLATE

# -----------------------------
# COUNTRY MODEL TOGGLE (edit here)
# -----------------------------
# Run A: COUNTRY_MODEL = ''  (no country model; single-stage)
# Run B: COUNTRY_MODEL = FLASH_COUNTRY_MODEL  (two-stage + KB)
FLASH_COUNTRY_MODEL = os.getenv('WORKSHOP_COUNTRY_MODEL', 'gemini-2.5-flash')
COUNTRY_MODEL = ''  # <-- EDIT THIS ONLY ('' or FLASH_COUNTRY_MODEL)

# -----------------------------
# FIXED PARAMS (keep unchanged for limited run)
# -----------------------------
NOTEBOOK_NAME = '04_advanced_parsing_with_knowledge'
MODEL_NAME = os.getenv('WORKSHOP_ADVANCED_MODEL', 'gemini-2.5-pro')

TEMPERATURE = 0.0
TOP_P = 0.95
TOP_K = 40
USE_GUARDRAILS = False  # present, but not a focus in this notebook

# Execution params (don't change)
MAX_TOKENS = preset_params["MAX_TOKENS"]
MAX_WORKERS = preset_params["MAX_WORKERS"]

# KB used only when running two-stage.
KB_CSV_PATH = str(PROJECT_ROOT / 'data/input/address_formats.csv')

# -----------------------------
# DATA (limited run)
# -----------------------------
dev_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/dev_labeled.csv', dtype=str).fillna('')
dev_small = dev_df.head(10).copy()

# -----------------------------
# STAGE SELECTION (automatic)
# -----------------------------
if COUNTRY_MODEL:
    STAGE_NAME = 'two_stage'
else:
    STAGE_NAME = 'advanced'

print('STAGE_NAME =', STAGE_NAME)
print('MODEL_NAME =', MODEL_NAME)
print('COUNTRY_MODEL =', COUNTRY_MODEL if COUNTRY_MODEL else '(none)')

# -----------------------------
# EXECUTE PIPELINE
# -----------------------------
t0 = time.perf_counter()

kwargs = dict(
    addresses=dev_small,
    email=STUDENT_EMAIL,
    stage=STAGE_NAME,
    model=MODEL_NAME,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    use_guardrails=USE_GUARDRAILS,
    custom_prompt_template=PROMPT_TO_RUN,
    max_workers=MAX_WORKERS,
)

# Only pass country_model + kb path when using two-stage.
if STAGE_NAME == 'two_stage':
    kwargs['country_model'] = COUNTRY_MODEL
    kwargs['kb_csv_path'] = KB_CSV_PATH

pred_df = process_batch_addresses(**kwargs)

runtime_sec = time.perf_counter() - t0
print('Runtime (sec):', round(runtime_sec, 3))

# Generate the report
report = evaluate_predictions(pred_df, dev_small, email=STUDENT_EMAIL)

# -----------------------------
# DISPLAY RESULTS (do not edit)
# -----------------------------
print(report['summary'])
display(report['field_metrics'])


## Step 2: Final Run on Full Dataset (your **publishable** score)

This is your **final run** on the full dev dataset.

1. Paste/confirm your best prompt in `STUDENT_PROMPT_TEMPLATE`.
2. Choose your final configuration:
   - **Single-stage:** set `COUNTRY_MODEL = ''` (runs `stage='advanced'`)
   - **Two-stage KB:** set `COUNTRY_MODEL = FLASH_COUNTRY_MODEL` (runs `stage='two_stage'`)

In this **full-run cell**, you may adjust any parameters you want (model choice, decoding params, workers, guardrails) since this run is used for publishing later.

In [ ]:
import time

# -----------------------------
# PROMPT (paste your best prompt here)
# -----------------------------
STUDENT_PROMPT_TEMPLATE = '''You are an AML address parser.

Input address:
{address}

Return strict JSON only with schema:
{schema}

Rules:
- Extract Town Name, Postal Code, Remaining Address, Country Code (2 characters).
- Town Name must be city/locality only.
- Postal Code must contain only postal tokens.
- Country Code must be ISO alpha-2 uppercase.
- Do not invent values.
- Use empty strings if unknown.
'''

PROMPT_TO_RUN = STUDENT_PROMPT_TEMPLATE

# -----------------------------
# COUNTRY MODEL TOGGLE (choose final approach)
# -----------------------------
FLASH_COUNTRY_MODEL = os.getenv('WORKSHOP_COUNTRY_MODEL', 'gemini-2.5-flash')
COUNTRY_MODEL = FLASH_COUNTRY_MODEL  # <-- set '' for single-stage, or FLASH_COUNTRY_MODEL for two-stage

# -----------------------------
# PARAMS (final run — you may adjust these)
# -----------------------------
NOTEBOOK_NAME = '04_advanced_parsing_with_knowledge'
MODEL_NAME = os.getenv('WORKSHOP_ADVANCED_MODEL', 'gemini-2.5-pro')

TEMPERATURE = 0.0
TOP_P = 0.95
TOP_K = 40
USE_GUARDRAILS = False

KB_CSV_PATH = str(PROJECT_ROOT / 'data/input/address_formats.csv')

# -----------------------------
# DATA (full run)
# -----------------------------
dev_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/dev_labeled.csv', dtype=str).fillna('')

# -----------------------------
# STAGE SELECTION (automatic)
# -----------------------------
if COUNTRY_MODEL:
    STAGE_NAME = 'two_stage'
else:
    STAGE_NAME = 'advanced'

print('STAGE_NAME =', STAGE_NAME)
print('MODEL_NAME =', MODEL_NAME)
print('COUNTRY_MODEL =', COUNTRY_MODEL if COUNTRY_MODEL else '(none)')

# -----------------------------
# EXECUTE PIPELINE
# -----------------------------
t0 = time.perf_counter()

kwargs = dict(
    addresses=dev_df,
    email=STUDENT_EMAIL,
    stage=STAGE_NAME,
    model=MODEL_NAME,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    use_guardrails=USE_GUARDRAILS,
    custom_prompt_template=PROMPT_TO_RUN,
    max_workers=MAX_WORKERS,
)

if STAGE_NAME == 'two_stage':
    kwargs['country_model'] = COUNTRY_MODEL
    kwargs['kb_csv_path'] = KB_CSV_PATH

pred_df = process_batch_addresses(**kwargs)

runtime_sec = time.perf_counter() - t0
print('Runtime (sec):', round(runtime_sec, 3))

# Report
report = evaluate_predictions(pred_df, dev_df, email=STUDENT_EMAIL)

print(report['summary'])
display(report['field_metrics'])


## Step 3: Placeholder — Publish to Leaderboard

This section will be wired to the leaderboard workflow later.

**Placeholder only for now.**

In [ ]:
publish_to_leaderboard(report, STUDENT_EMAIL)